# Using bvhTools in a ML pipeline
In this section, we will show an example of preparing a BVHDatasetView object to use it in out Torch ML pipeline. We will use a subset of the [Ubisoft's LAFAN1 dataset](https://github.com/ubisoft/ubisoft-laforge-animation-dataset/) for the example. We will make a train set, a validation set and a test set. We will set a 6D internal representation for the angles, and we will normalize all the data. We will make 3 partitions, and windows of 128 frames with a stride of 64, and we will create and set sequence labels to describe the action in each file.

First, we will start downloading the data from [here](https://github.com/ubisoft/ubisoft-laforge-animation-dataset/). Locate the lafan.zip file in the repository and download it. Then, unzip it to a new folder called `lafan1` and locate it in this tutorials folder.

## 1) Creating the dataset
Now that we have the files, we are going to load all the files and create a new `BVHDataset` object, using the `from_folder` function, since we want to bulk load all the files. In this case, the lafan1 dataset contains 77 BVH files.

In [1]:
from bvhTools.bvhML import BVHDataset
dataset = BVHDataset.fromFolder("./lafan1")
print(len(dataset))

77


### Creating and attaching the labels
Now, we are going to create a label yaml file. To simplify the process, since the `lafan1` dataset contains the type of the animation in the animation file itself, we will use the first part of the file name as label. We are going to make a small python script that goes over the entire folder, and for each file, it uses the relative path as key and creates a sequence label with the motion type. This process needs to be done just once, and the yaml file can be created as the user needs, not specifically like this; hovewer it is important to follow the yaml format.

In [2]:
import yaml
import os

labels = {}
for file in os.listdir("./lafan1"):
    if file.endswith(".bvh"):
        abs_path = os.path.join("./lafan1", file)
        label = file.split("_")[0]
        labels[abs_path] = {
            "sequence": {
                "action": label
            }
        }

with open("labels.yaml", "w") as f:
    yaml.dump(labels, f, sort_keys=False)

This code will create a `labels.yaml` file that we can now load and attach to our dataset object. This way, each file of our dataset will contain a sequence label, named `action` which will contain the name of the action. We can attach the labels to the original dataset object.

In [3]:
dataset.attachLabels("labels.yaml")

### Creating the splits
Now that we have attached our labels, we can define the data splits before we can start making the BVHDatasetView class. This way, we will connect each file to a specific split, so there is no cross-contamination between splits. Papers that use lafan1 usually set `subject5` as the testing subject. We will use `subject4` as the validation subject for example. In our case, the easies to do the splits is to use patterns. We will make a new yaml file named `splits.yaml` that will look like this. 

In [4]:
with open("splits.yaml", "w") as f:
    f.write("splitRule: pattern\n\
splits:\n\
    train:\n\
        - subject1\n\
        - subject2\n\
        - subject3\n\
    validation:\n\
        - subject4\n\
    test:\n\
        - subject5")

Now, we will load this yaml file and define the splits. This way, we have already finished constructing our BVHDataset class.

In [5]:
dataset.defineSplits("splits.yaml")

Now, we have finished constructing the BVHDataset class, which represents the collection of all the loaded files, with their respective labels and with defined splits. The following step is to create the  dataset view, which is windowed, and has a useful representation rather than Euler angles, so a machine learning algorith can load many animation pieces in a batch and use them for training.

## 2) Creating the dataset view
We are going to use a window size of 128 and a stride of 64, for example. Moreover, we are going to use a Euler representation in the example, because it will not need to do any angle conversion and it wil be much faster. However, you can try other representations like 6D representation, which is currently the state of the art. If you test it by providing `representation = "sixd"`, you will notice that this process might be slighly slow, since changing the format of the angles is a cpu intensive task. The representation can be changed after, but it is not recommended, for this reason.

**Note: However, since bvhTools saves a cache of the precomputed representation, if you make another view of the same dataset with the same representation, the process will be very fast.**

In [6]:
datasetView = dataset.view(windowLength = 128, stride = 64, representation = "euler")

We can inspect what we have created by showing the size of the created view, and by printing one instance of the dataset view. We can see that the dataset now has 7639 motion windows, each one of shape (128,135), meaning that each sequence is 128 frames long, and each frame has 135 elements (3 positions + 6 * 22 rotations). The instance also contains the sequence label, and no temporal labels.

In [7]:
print(len(datasetView))
print(datasetView[42]["motion"].shape)
print(datasetView[42]["labels"]["sequence"])
print(datasetView[42]["labels"]["temporal"])

7639
(128, 69)
{'action': 'aiming1'}
None


### Normalizing the data
After creating the view, since the internal angle representation is already defined, we are able to compute normalization statistics to normalize the data. We use the `normalize` function to compute this statistics. For this example we will use z-score normalization. We will show a small part of the normalized data and compare it to the denormalized data.

In [8]:
print("Pre-normalized motion snippet")
print(datasetView[42]["motion"][0:10])
datasetView.normalize(mode = "zscore")
print("Normalized motion snippet")
print(datasetView[42]["motion"][0:10])

Pre-normalized motion snippet
[[ 2.83148590e+02  8.99048460e+01 -4.51795410e+02  8.63689580e+01
  -6.65131300e+00 -5.96271960e+01  1.79098516e+02  8.24839800e+00
  -1.78824393e+02 -1.75327370e+01 -4.54810100e+00 -5.22881800e+00
   7.58042470e+01  6.51485000e+00  9.74324600e+00  2.14545570e+01
   3.10000000e-03  1.00000000e-06  1.62402856e+02  7.92609700e+00
  -1.72532136e+02 -7.33669890e+01 -2.10019100e+00  5.61806500e+00
   6.98238560e+01 -3.16478200e+00  2.40252700e+00  2.14545600e+01
  -3.10900000e-03 -1.50000000e-05  7.78219400e+00 -1.43416900e+00
   1.87674000e-01  5.54942800e+00 -2.87568500e+00  3.06823000e-01
   5.23088800e+00 -2.87363500e+00  3.24206000e-01  9.53215000e+00
   1.83273990e+01  1.50584310e+01 -1.33793720e+01 -8.59746200e+00
   2.00767040e+01  1.54067307e+02 -7.34531380e+01  2.50973640e+01
  -3.68540070e+01  6.01755650e+01 -5.82301900e+00 -5.01358120e+01
  -1.15176110e+01  1.52253100e+00  1.48211720e+01  3.60047500e+00
  -8.01367720e+01 -1.61752007e+02  7.44875160e

Since this normalization involves going through all the frames in the dataset, we usually don't want to repeat the process. For this reason, we will write the calculated statistics to a file, and we will load them the next time that we want to normalize the data to avoid computation. We will use a `numpy` file since we are writing numpy arrays, but it can be saved and loaded as desired, taking into account that the normalization mode is a string and normalization statistics is a dictionary with numpy arrays.

In [9]:
normalizationMode, normalizationStatistics = datasetView.getNormalizationStatistics()
data = {
    'normalizationMode': normalizationMode,
    'normalizationStatistics': normalizationStatistics
}

with open("normalization.yaml", "w") as f:
    yaml.dump(data, f)

We can load the statistics and use them by loading the yaml file.

In [10]:
with open("normalization.yaml", "r") as f:
    data = yaml.safe_load(f)

normalizationMode = data["normalizationMode"]
normalizationStatistics = data["normalizationStatistics"]
datasetView.setNormalizationStatistics(normalizationMode, normalizationStatistics)

### Creating or selecting the active split
Finally, let's say that we want to use the dataset object just for a training process. We can just set the "train" split as the active split, and the `BVHDatasetView` object will behave as it just contained the train data. The size of the object will be the size of the training set, and the sequences from "validation" and "test" will be ignored.

In [11]:
datasetView.setSplit("train")

If, on the other hand, we wanted to make the three splits, we could us the `makeSplit` function and create new `BVHDatasetView` objects, each one representing one split. Take into account that these new objects are copies of the original one, and contain the reference to the original `BVHDataset` class. This means that they won't need much space in disk, but be careful, if you edit the original object, these new partitions can break.

In [12]:
trainSplit = datasetView.makeSplit("train")
valSplit = datasetView.makeSplit("validation")
testSplit = datasetView.makeSplit("test")

print(len(trainSplit))
print(len(valSplit))
print(len(testSplit))

4763
1492
1384


## 3) [OPTIONAL] materializing the data
If you are interested in loading all the sequences or windows that you have created in memory (i.e. creating all the small pieces and having them as numpmy arrays in memory), you can materialize the dataset. This will basically use the original files with the original motion to create new copies of the windows and they will be loaded in memory. Take into account that if your dataset is big and you have made a lot of windows, this can take a very big space in RAM. The materialize method will return a `BVHDatasetViewMaterialized` object, you can look the specific details of this object in the documentation.

In [13]:
trainSplitMaterialized = trainSplit.materialize()
valSplitMaterialized = valSplit.materialize()
testSplitMaterialized = testSplit.materialize()

print(len(trainSplitMaterialized))
print(len(valSplitMaterialized))
print(len(testSplitMaterialized))

4763
1492
1384


## SUMMARY OF THE CODE

In [14]:
from bvhTools.bvhML import BVHDataset

# Load the data from the folder
dataset = BVHDataset.fromFolder("./lafan1")
# Attach the labels from a yaml file
dataset.attachLabels("labels.yaml")
# Create train/val/test splits from a yaml file
dataset.defineSplits("splits.yaml")
# Window the dataset and create a view with a specific rotation representation (try other representations)
datasetView = dataset.view(windowLength = 128, stride = 64, representation = "euler")
# Calculate the normalization stats (or load them from file)
datasetView.normalize(mode = "zscore")
# Set the active split
datasetView.setSplit("train")